<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

# 경제 예제: 사업 선택

## 1) 예제 설명

- 예산 7백만 원 안에서 사업을 골라 순편익(net benefit) 최대화
- 사업별 효과를 더할 수 있다고 가정, 가상 자료의 단위는 백만 원

| 사업 | 비용 | 순편익 |
|:---|:---|:---|
| A | 4 | 7 |
| B | 3 | 6 |
| C | 2 | 4 |
| D | 5 | 8 |

- 의사결정벡터(decision vector): $\mathbf{x}=(x_1,x_2,x_3,x_4)$, 각 성분은 A·B·C·D의 실행 여부, 1은 실행·0은 제외
- 입자 위치: 탐색에 사용하는 실수 벡터 $\mathbf{z}=(z_1,z_2,z_3,z_4)$, 각 성분의 범위는 0~1
- 목적함수(objective function): 선택한 사업의 순편익 합, 비용은 순편익에 이미 반영되어 다시 차감하지 않음
- 제약(constraint): 총비용이 예산 7 이하여야 허용, 초과 후보는 순편익 비교에서 제외

- A·C의 비용은 6으로 허용, A·D의 비용은 9로 거부


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 2) 구현

- 공통 PSO 함수는 탐색 후 전역 최적 위치 gbest 반환

- 초기 속도의 각 성분은 해당 탐색 구간 폭의 −10%~10%에서 균등분포 난수로 설정


In [ ]:
import numpy as np


def particle_swarm(initial_positions, objective, lower, upper, iterations=20, seed=0):
    # 난수 생성기와 초기 위치 설정
    rng = np.random.default_rng(seed)
    positions = np.array(initial_positions, dtype=float, copy=True)
    if np.any(positions < lower) or np.any(positions > upper):
        raise ValueError("Initial positions must satisfy the bounds.")

    # 탐색 구간 폭에 비례하는 초기 속도 설정
    span = np.asarray(upper) - np.asarray(lower)
    velocities = rng.uniform(-0.1, 0.1, positions.shape) * span

    # 초기 위치를 평가하여 개인 및 전역 최적 위치 설정
    pbest = positions.copy()
    best_values = np.array([objective(x) for x in positions])
    gbest = pbest[np.argmin(best_values)].copy()

    for _ in range(iterations):
        # 관성항, 인지항, 사회항으로 속도 갱신
        r1 = rng.random(positions.shape)
        r2 = rng.random(positions.shape)
        inertia = 0.6 * velocities
        cognitive_term = 1.4 * r1 * (pbest - positions)
        social_term = 1.4 * r2 * (gbest - positions)
        velocities = inertia + cognitive_term + social_term

        # 위치 이동 후 경계값으로 제한, 벗어난 성분의 속도는 0으로 설정
        proposed = positions + velocities
        positions = np.clip(proposed, lower, upper)
        velocities[positions != proposed] = 0.0

        # 이동한 위치의 목적함수값 계산
        values = np.array([objective(x) for x in positions])

        # 기존 기록보다 좋은 위치로 개인 최적 위치 갱신
        improved = values < best_values
        pbest[improved] = positions[improved]
        best_values[improved] = values[improved]

        # 모든 개인 최적 위치 중 가장 좋은 위치 선택
        gbest = pbest[np.argmin(best_values)].copy()

    # 반복 종료 후 전역 최적 위치 반환
    return gbest


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 3) 예제

- 공통 PSO 함수에 사업 자료와 예산 검사·평가 함수를 전달, 20회 반복 후 사업 선택·비용·순편익 출력
- 예산 초과는 ‘inf’로 평가하여 최선 기록에서 제외, 초기 후보 네 개는 모두 예산 충족


In [ ]:
# 사업별 비용, 순편익 및 예산
costs = [4, 3, 2, 5]
benefits = [7, 6, 4, 8]
budget = 7

# 실수 위치를 사업 선택 여부로 변환
def decode(position):
    return (np.asarray(position) >= 0.5).astype(int)

# 예산 충족 여부를 확인한 뒤 순편익 계산
def evaluate(decision):
    cost = sum(costs[i] * decision[i] for i in range(4))
    if cost > budget:
        return {"cost": cost, "feasible": False, "net_benefit": None}

    benefit = sum(benefits[i] * decision[i] for i in range(4))
    return {"cost": cost, "feasible": True, "net_benefit": benefit}

# 예산 초과 후보는 제외하고 순편익의 음수를 최소화
def objective(position):
    result = evaluate(decode(position))
    if not result["feasible"]:
        return float("inf")
    return -result["net_benefit"]

# 초기 후보 설정 및 PSO 실행
initial_positions = [[1, 0, 0, 0], [0, 1, 0, 0],
                     [0, 0, 1, 0], [0, 0, 0, 1]]
result = particle_swarm(initial_positions, objective, 0, 1, iterations=20)

# 탐색 결과를 실제 의사결정으로 변환하고 출력
decision = decode(result)
evaluation = evaluate(decision)
print("Decision:", decision)
print("Cost (million KRW):", evaluation["cost"])
print("Feasible:", evaluation["feasible"])
print("Net benefit (million KRW):", evaluation["net_benefit"])
print("Stop reason: iteration limit")
